### Step 0: Start the Server (if making HTTP calls)
In a terminal, start the FastAPI server:


powershell
>```powershell
>.venv\Scripts\uvicorn src.api.app:app --reload --port 8000
>```

*(The API will be available at http://localhost:8000).*

### Option 1: Python HTTP Request (Recommended for testing.ipynb)
Using requests (or httpx) to send a transaction to /v1/predict matching the TransactionPayload schema:

In [ ]:
import requests
url = "http://localhost:8000/v1/predict"
# Example: Suspicious transaction (large TRANSFER that completely empties the account)
payload = {
    "step": 200,
    "type": "TRANSFER",
    "amount": 450000.0,
    "nameOrig": "C123456789",
    "oldbalanceOrg": 450000.0,
    "newbalanceOrig": 0.0,
    "nameDest": "C987654321",
    "oldbalanceDest": 0.0,
    "newbalanceDest": 0.0,
}
response = requests.post(url, json=payload)
if response.status_code == 200:
    result = response.json()
    print(f"Decision:    {result['decision']}")
    print(f"Risk Tier:   {result['risk_tier']}")
    print(f"Risk Score:  {result['risk_score']}/1000")
    print(f"Probability: {result['fraud_probability']:.4f}")
    print("\nTop SHAP Decision Factors:")
    for factor in result["top_factors"]:
        print(f" - {factor['feature']} (val: {factor['value']}): impact {factor['shap_impact']:+.4f} ({factor['direction']})")
else:
    print(f"Error {response.status_code}:", response.text)


### Option 2: In-Memory via TestClient (No Server Required!)
If you are working inside testing.ipynb and do not want to run uvicorn in a separate terminal window, you can invoke the FastAPI app directly in memory using Starlette's TestClient:

In [ ]:
from fastapi.testclient import TestClient
from src.api.app import app
client = TestClient(app)
# Legitimate payment example
payload = {
    "step": 50,
    "type": "PAYMENT",
    "amount": 34.50,
    "nameOrig": "C1029384",
    "oldbalanceOrg": 1200.0,
    "newbalanceOrig": 1165.50,
    "nameDest": "M9281726",
    "oldbalanceDest": 0.0,
    "newbalanceDest": 0.0,
}
response = client.post("/v1/predict", json=payload)
print(response.json())


### Option 3: Terminal / cURL / PowerShell
Using PowerShell:

```powershell
$body = @{
    step = 200
    type = "TRANSFER"
    amount = 500000.0
    nameOrig = "C1928371"
    oldbalanceOrg = 500000.0
    newbalanceOrig = 0.0
    nameDest = "C9981273"
    oldbalanceDest = 0.0
    newbalanceDest = 0.0
} | ConvertTo-Json
Invoke-RestMethod -Uri "http://localhost:8000/v1/predict" -Method Post -ContentType "application/json" -Body $body
```

Using curl:

```bash
curl -X POST http://localhost:8000/v1/predict \
  -H "Content-Type: application/json" \
  -d '{
    "step": 200,
    "type": "TRANSFER",
    "amount": 500000.0,
    "nameOrig": "C1928371",
    "oldbalanceOrg": 500000.0,
    "newbalanceOrig": 0.0,
    "nameDest": "C9981273",
    "oldbalanceDest": 0.0,
    "newbalanceDest": 0.0
  }'
```

Expected Response
The endpoint returns a PredictionResponse payload formatted like:

```json
{
  "fraud_probability": 0.9998,
  "risk_score": 999,
  "risk_tier": "CRITICAL",
  "decision": "DECLINE",
  "threshold_applied": 0.02,
  "top_factors": [
    {
      "feature": "orig_balance_drain",
      "value": 1.0,
      "shap_impact": 4.12,
      "direction": "RISK_INCREASING"
    },
    {
      "feature": "type_TRANSFER",
      "value": 1.0,
      "shap_impact": 2.85,
      "direction": "RISK_INCREASING"
    }
  ]
}
```